# 🔥 Fine-Tune Qwen 2.5 1.5B-Instruct untuk Asisten K3

| Item | Detail |
|------|--------|
| Base Model | `Qwen/Qwen2.5-1.5B-Instruct` |
| Method | LoRA (r=16, alpha=32) |
| Platform | Kaggle (2x T4 GPU) |
| Output | `qwen2.5-1.5b-k3.gguf` (~950MB) |

> ⚠️ **Setelah Cell 1 selesai, WAJIB Restart Session (Ctrl+M+.) lalu lanjut Cell 2.**

## 1. Install Dependencies

In [ ]:
!pip install -q "trl>=0.17" "transformers>=4.51" "peft>=0.15" \
    datasets accelerate bitsandbytes sentencepiece \
    huggingface-hub safetensors tokenizers gguf

import torch
n_gpu = torch.cuda.device_count()
print(f'\nTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()}, GPU: {n_gpu}x')
for i in range(n_gpu):
    name = torch.cuda.get_device_name(i)
    mem = torch.cuda.get_device_properties(i).total_mem / 1024**3
    print(f'  [{i}] {name} ({mem:.1f} GB)')
assert torch.cuda.is_available(), '❌ CUDA tidak tersedia!'
print('\n✅ OK! WAJIB: Restart Session (Ctrl+M+.) lalu lanjut Cell 2')

## 2. Login Hugging Face

In [ ]:
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    login(token=UserSecretsClient().get_secret('HF_TOKEN'))
    print('✅ Login via Kaggle Secrets!')
except:
    try:
        from google.colab import userdata
        login(token=userdata.get('HF_TOKEN'))
        print('✅ Login via Colab Secrets!')
    except:
        token = input('Paste HF Token: ').strip()
        login(token=token)
        print('✅ Login berhasil!')

## 3. Download / Detect Dataset

In [ ]:
import os, glob

# Coba download dari R2/CDN dulu
DATASET_URL = 'https://data.scz.my.id/dataset_100k.jsonl'
!wget -q -nc {DATASET_URL}

# Auto-detect
DATASET_PATH = None
for pattern in ['/kaggle/working/*.jsonl', '/kaggle/input/**/*.jsonl', '/content/*.jsonl']:
    found = glob.glob(pattern, recursive=True)
    if found:
        DATASET_PATH = found[0]
        break

assert DATASET_PATH, '❌ Dataset .jsonl tidak ditemukan!'
print(f'✅ Dataset: {DATASET_PATH}')

## 4. Format Dataset (ChatML untuk Qwen)

In [ ]:
import json
from datasets import Dataset

data = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line.strip()))

SYS_PROMPT = (
    'Kamu adalah Asisten K3 (Keselamatan dan Kesehatan Kerja) Indonesia. '
    'Jawab dengan akurat, singkat, dan informatif dalam Bahasa Indonesia.'
)

def format_chatml(item):
    """Format ChatML khusus Qwen."""
    q = item['instruction']
    a = item['response']
    # Truncate response panjang agar training tidak lambat
    if len(a) > 1500:
        a = a[:1500] + '...'
    text = (
        f'<|im_start|>system\n{SYS_PROMPT}<|im_end|>\n'
        f'<|im_start|>user\n{q}<|im_end|>\n'
        f'<|im_start|>assistant\n{a}<|im_end|>'
    )
    return {'text': text}

dataset = Dataset.from_list(data).map(format_chatml)
split = dataset.train_test_split(test_size=0.05, seed=42)
train_ds = split['train']
eval_ds = split['test']

print(f'✅ Dataset: {len(train_ds)} train, {len(eval_ds)} eval')
print(f'\nContoh:\n{train_ds[0]["text"][:400]}...')

## 5. Load Model + LoRA

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map='auto',
    torch_dtype=torch.float16,
    attn_implementation='sdpa',
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()
print(f'GPU Memory: {torch.cuda.memory_allocated()/1024**2:.0f} MB')
print('\n✅ Model Qwen siap!')

## 6. Training

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import torch

n_gpu = torch.cuda.device_count()
per_device_batch = 2
grad_accum = 16
effective_batch = per_device_batch * n_gpu * grad_accum
n_epochs = 1
total_steps = (len(train_ds) * n_epochs) // effective_batch
warmup_steps = int(total_steps * 0.05)

print(f'GPUs: {n_gpu}, Batch/GPU: {per_device_batch}, '
      f'Grad Accum: {grad_accum}, Effective Batch: {effective_batch}')
print(f'Epochs: {n_epochs}, Total Steps: ~{total_steps}, Warmup: {warmup_steps}')

training_args = TrainingArguments(
    output_dir='/kaggle/working/k3_checkpoints',
    num_train_epochs=n_epochs,
    per_device_train_batch_size=per_device_batch,
    per_device_eval_batch_size=per_device_batch,
    gradient_accumulation_steps=grad_accum,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=warmup_steps,
    weight_decay=0.01,
    fp16=True,
    logging_steps=25,
    eval_strategy='steps',
    eval_steps=500,
    save_strategy='steps',
    save_steps=500,
    save_total_limit=2,
    report_to='none',
    seed=42,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
)

print('Training dimulai...')
trainer.train()
print('\n✅ Training selesai!')
trainer.save_model('/kaggle/working/k3_lora')
print('✅ LoRA adapter tersimpan!')

## 7. Merge LoRA + Convert ke GGUF

In [ ]:
import subprocess, os, shutil, glob

MODEL_ID   = 'Qwen/Qwen2.5-1.5B-Instruct'
MERGED_DIR = '/kaggle/working/k3_merged'
GGUF_PATH  = '/kaggle/working/qwen2.5-1.5b-k3.gguf'
LLAMA_DIR  = '/kaggle/working/llama_cpp_repo'

# ── 1. Merge LoRA ──
if not os.path.exists(os.path.join(MERGED_DIR, 'model.safetensors')):
    print('📦 Merging LoRA weights...')
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(MERGED_DIR)
    print('✅ Weights merged!')
else:
    print('✅ Merged model sudah ada, skip.')

# ── 2. Restore tokenizer ORIGINAL ──
from transformers import AutoTokenizer
orig_tok = AutoTokenizer.from_pretrained(MODEL_ID)
orig_tok.save_pretrained(MERGED_DIR)
added = os.path.join(MERGED_DIR, 'added_tokens.json')
if os.path.exists(added): os.remove(added)
print('✅ Original tokenizer restored!')

# ── 3. Clone llama.cpp ──
if os.path.exists(LLAMA_DIR): shutil.rmtree(LLAMA_DIR)
subprocess.run(['git', 'clone', '--depth', '1',
                'https://github.com/ggerganov/llama.cpp.git', LLAMA_DIR],
               capture_output=True, check=True)
subprocess.run(['pip', 'install', '-q', 'gguf'], capture_output=True, check=True)
print('✅ llama.cpp cloned!')

# ── 4. Convert HF → GGUF F16 (Qwen tidak perlu patch vocab) ──
print('\n🔄 Converting ke GGUF F16...')
script = f'{LLAMA_DIR}/convert_hf_to_gguf.py'
result = subprocess.run(
    ['python', script, MERGED_DIR,
     '--outfile', '/kaggle/working/model-f16.gguf', '--outtype', 'f16'],
    capture_output=True, text=True
)
if result.returncode != 0:
    print('❌ STDERR:', result.stderr[-2000:])
    raise RuntimeError('Konversi GGUF gagal!')
print('✅ F16 GGUF berhasil!')

# ── 5. Build llama-quantize ──
print('🔨 Building llama-quantize...')
subprocess.run(['cmake', '-B', 'build', '-DGGML_CUDA=OFF'],
               cwd=LLAMA_DIR, capture_output=True, check=True)
subprocess.run(['cmake', '--build', 'build', '--config', 'Release', '-j4',
                '--target', 'llama-quantize'],
               cwd=LLAMA_DIR, capture_output=True, check=True)
qbin = glob.glob(f'{LLAMA_DIR}/build/**/llama-quantize', recursive=True)
if not qbin:
    qbin = glob.glob(f'{LLAMA_DIR}/build/**/quantize', recursive=True)
assert qbin, 'llama-quantize tidak ditemukan!'
qbin = qbin[0]
print(f'✅ Build selesai: {qbin}')

# ── 6. Quantize F16 → Q4_K_M ──
print('🔄 Quantizing ke Q4_K_M...')
subprocess.run([qbin, '/kaggle/working/model-f16.gguf', GGUF_PATH, 'Q4_K_M'], check=True)

size_mb = os.path.getsize(GGUF_PATH) / 1024**2
print(f'\n🎉 SELESAI! GGUF: {GGUF_PATH} ({size_mb:.1f} MB)')

## 8. Quick Test

In [ ]:
!pip install -q llama-cpp-python

from llama_cpp import Llama

GGUF_PATH = '/kaggle/working/qwen2.5-1.5b-k3.gguf'
m = Llama(model_path=GGUF_PATH, n_ctx=1024, n_threads=4, verbose=False)
print('✅ Model loaded!\n')

SYS = 'Kamu adalah Asisten K3. Jawab dalam Bahasa Indonesia.'

test_questions = [
    'Apa itu APAR dan bagaimana cara menggunakannya?',
    'Apa yang harus dilakukan saat terjadi kebakaran di gedung?',
    'Sebutkan kelas-kelas kebakaran!',
]

for q in test_questions:
    # FORMAT CHATML UNTUK QWEN
    prompt = (
        f'<|im_start|>system\n{SYS}<|im_end|>\n'
        f'<|im_start|>user\n{q}<|im_end|>\n'
        f'<|im_start|>assistant\n'
    )
    r = m(prompt, max_tokens=200, temperature=0.3,
          stop=['<|im_end|>', '<|endoftext|>'])
    answer = r['choices'][0]['text'].strip()
    print(f'Q: {q}')
    print(f'A: {answer}\n')
    print('-' * 60)

## 9. Upload ke Cloudflare R2

In [ ]:
!pip install -q boto3

import os, boto3

GGUF_PATH = '/kaggle/working/qwen2.5-1.5b-k3.gguf'
size_mb = os.path.getsize(GGUF_PATH) / 1024**2
print(f'File: {GGUF_PATH} ({size_mb:.1f} MB)')

# ==========================================
# KONFIGURASI CLOUDFLARE R2
# ==========================================
R2_ACCOUNT_ID = 'GANTI_DENGAN_ACCOUNT_ID'
R2_ACCESS_KEY = 'GANTI_DENGAN_ACCESS_KEY'
R2_SECRET_KEY = 'GANTI_DENGAN_SECRET_KEY'
BUCKET_NAME   = 'GANTI_DENGAN_NAMA_BUCKET'
OBJECT_NAME   = 'qwen2.5-1.5b-k3.gguf'
# ==========================================

if 'GANTI' in R2_ACCOUNT_ID:
    print('⚠️ Upload dilewati: isi kredensial R2 terlebih dahulu.')
    print('💾 Kaggle: Buka tab Output di sidebar kanan untuk download manual.')
else:
    print(f'☁️ Upload ke R2 ({BUCKET_NAME})...')
    s3 = boto3.client('s3',
        endpoint_url=f'https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com',
        aws_access_key_id=R2_ACCESS_KEY,
        aws_secret_access_key=R2_SECRET_KEY,
    )
    try:
        s3.upload_file(GGUF_PATH, BUCKET_NAME, OBJECT_NAME)
        print('✅ Upload berhasil!')
        url = s3.generate_presigned_url(
            'get_object',
            Params={'Bucket': BUCKET_NAME, 'Key': OBJECT_NAME},
            ExpiresIn=3600,
        )
        print(f'\n🔗 Download URL (1 jam): {url}')
    except Exception as e:
        print(f'❌ Upload gagal: {e}')